# Phân tích Doanh số Pakistan E-Commerce — Spark SQL

# Setup

## Khởi tạo SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("SQL") \
    .getOrCreate()

HDFS_PATH = "hdfs://localhost:9000/ecom/Pakistan_Largest_Ecommerce_Dataset.csv"

df = spark.read.csv(HDFS_PATH, header=True, inferSchema=False)

df.show(5, truncate=False)
df.printSchema()

+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|item_id|status        |created_at|sku                                                        |price|qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|Working Date|BI Status| MV    |Year|Month|Customer Since|M-Y   |FY  |Customer ID|_c21|_c22|_c23|_c24|_c25|
+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|211131 |complete      |7/1/2016  |kreations_YI 06-L

## Đọc dữ liệu sạch từ HDFS

In [2]:
df = spark.read.csv(
    "hdfs://localhost:9000/ecom/ecom_clean_final",
    header=True,
    inferSchema=True
)

print(f"   Số dòng : {df.count():,}")
print(f"   Số cột  : {len(df.columns)}")
df.printSchema()

   Số dòng : 429,296
   Số cột  : 16
root
 |-- item_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty_ordered: double (nullable = true)
 |-- grand_total: double (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- category_name_1: string (nullable = true)
 |-- sales_commission_code: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)



## Kiểm tra các giá trị phân loại

In [3]:
print("Các giá trị status")
df.groupBy("status").count().orderBy(F.desc("count")).show(20, truncate=False)

print("Top 10 category_name_1")
df.groupBy("category_name_1").count().orderBy(F.desc("count")).show(10, truncate=False)

Các giá trị status
+--------------+------+
|status        |count |
+--------------+------+
|complete      |191172|
|canceled      |123348|
|received      |57314 |
|order_refunded|46946 |
|refund        |6894  |
|cod           |2383  |
|paid          |748   |
|closed        |406   |
|processing    |28    |
|payment_review|22    |
|pending       |15    |
|holded        |10    |
|pending_paypal|6     |
|exchange      |4     |
+--------------+------+

Top 10 category_name_1
+-----------------+-----+
|category_name_1  |count|
+-----------------+-----+
|Men's Fashion    |85266|
|Mobiles & Tablets|72940|
|Women's Fashion  |54649|
|Appliances       |38332|
|Beauty & Grooming|36378|
|Soghaat          |27294|
|Superstore       |25514|
|Home & Living    |23092|
|Health & Sports  |14728|
|Kids & Baby      |14568|
+-----------------+-----+
only showing top 10 rows


## CATALOG — Temp View `sales`

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `increment_id` | string | Mã đơn hàng (unique per order) |
| `item_id` | string | Mã dòng sản phẩm (một đơn có thể có nhiều item) |
| `customer_id` | string | Mã khách hàng |
| `category_name_1` | string | Danh mục sản phẩm cấp 1 |
| `sku` | string | Mã SKU sản phẩm |
| `price` | double | Đơn giá sản phẩm |
| `qty_ordered` | double | Số lượng đặt hàng |
| `grand_total` | double | Tổng tiền thực thu (sau giảm giá) |
| `discount_amount` | double | Số tiền giảm giá; `grand_total + discount_amount` = giá gốc |
| `payment_method` | string | Phương thức thanh toán (COD, easypay, ...) |
| `status` | string | Trạng thái đơn hàng (`complete`, `canceled`, `order_refunded`, ...) |
| `order_year` | integer | Năm đặt hàng (có thể NULL) |
| `order_month` | integer | Tháng đặt hàng (có thể NULL) |


In [4]:
sales = df.select(
    "increment_id", "item_id", "customer_id",
    "category_name_1", "sku",
    "price", "qty_ordered", "grand_total", "discount_amount",
    "payment_method", "status",
    "order_year", "order_month"
)
sales.createOrReplaceTempView("sales")

spark.sql("SELECT * FROM sales LIMIT 5").show(truncate=True)

+------------+-------+-----------+-----------------+--------------------+------+-----------+-----------+---------------+--------------+--------+----------+-----------+
|increment_id|item_id|customer_id|  category_name_1|                 sku| price|qty_ordered|grand_total|discount_amount|payment_method|  status|order_year|order_month|
+------------+-------+-----------+-----------------+--------------------+------+-----------+-----------+---------------+--------------+--------+----------+-----------+
|   100147569| 211313|         57|          Soghaat|       RS_Kaju Barfi| 425.0|        1.0|     1125.0|            0.0|           cod|complete|      2016|          7|
|   100147818| 211630|        147|          Soghaat|   HR_Pani Puri 360g| 350.0|        1.0|      350.0|            0.0|           cod|complete|      2016|          7|
|   100148140| 212180|        341|          Soghaat|UK_Namkino Khat M...|  80.0|        1.0|      240.0|            0.0|           cod|canceled|      2016|     

---
# Truy vấn Spark SQL — Insight bán hàng

## Câu 1: Doanh thu và sản lượng theo danh mục sản phẩm
**Truy vấn:** Aggregation (SUM, COUNT, AVG) + GROUP BY + ORDER BY  
**Insight:** Danh mục nào tạo ra doanh thu cao nhất / thấp nhất, danh mục nào bán được nhiều đơn nhất -
cơ sở để ưu tiên nhập hàng và phân bổ ngân sách marketing.


In [5]:
result1 = spark.sql("""
    SELECT
        category_name_1                         AS category,
        COUNT(DISTINCT increment_id)            AS total_orders,
        SUM(qty_ordered)                        AS total_qty_sold,
        ROUND(SUM(grand_total), 2)              AS total_revenue,
        ROUND(AVG(grand_total), 2)              AS avg_order_value
    FROM sales
    GROUP BY category_name_1
    ORDER BY total_revenue DESC
""")
result1.show(20, truncate=False)

+------------------+------------+--------------+--------------+---------------+
|category          |total_orders|total_qty_sold|total_revenue |avg_order_value|
+------------------+------------+--------------+--------------+---------------+
|Mobiles & Tablets |67409       |72940.0       |4.1995425385E8|5757.53        |
|Women's Fashion   |36858       |54649.0       |1.7746154926E8|3247.3         |
|Appliances        |35466       |38332.0       |1.7369520735E8|4531.34        |
|Men's Fashion     |63499       |85266.0       |1.4139438029E8|1658.27        |
|Entertainment     |10515       |10909.0       |8.150747823E7 |7471.58        |
|Beauty & Grooming |26280       |36378.0       |6.521901971E7 |1792.81        |
|Home & Living     |16384       |23092.0       |4.925618266E7 |2133.04        |
|Superstore        |15216       |25514.0       |4.452023723E7 |1744.93        |
|Computing         |11597       |12363.0       |3.685860524E7 |2981.36        |
|Kids & Baby       |10030       |14568.0

## Câu 2: Top 10 sản phẩm (SKU) bán chạy nhất theo doanh thu
**Truy vấn:** Aggregation + GROUP BY + HAVING + LIMIT  
**Insight:** Xác định SKU nào đóng góp doanh thu lớn nhất - ưu tiên giữ tồn kho,
đẩy mạnh quảng cáo cho các sản phẩm này.


In [6]:
result2 = spark.sql("""
    SELECT
        sku,
        category_name_1                         AS category,
        COUNT(DISTINCT increment_id)            AS total_orders,
        SUM(qty_ordered)                        AS total_qty_sold,
        ROUND(SUM(grand_total), 2)              AS total_revenue
    FROM sales
    GROUP BY sku, category_name_1
    HAVING SUM(qty_ordered) > 0
    ORDER BY total_revenue DESC
    LIMIT 10
""")
result2.show(10, truncate=False)

+-------------------+-----------------+------------+--------------+-------------+
|sku                |category         |total_orders|total_qty_sold|total_revenue|
+-------------------+-----------------+------------+--------------+-------------+
|MATSAM59DB75ADB2F80|Mobiles & Tablets|3751        |3751.0        |4.303440278E7|
|MATSAM5A7463EE3C1A5|Mobiles & Tablets|815         |815.0         |1.166870895E7|
|Infinix Hot 4-Gold |Mobiles & Tablets|913         |913.0         |1.090553412E7|
|ENTNOB5A4633C950FAD|Entertainment    |767         |767.0         |1.07068618E7 |
|ENTNOB5A14947A21475|Entertainment    |699         |699.0         |9669446.71   |
|Infinix Hot 4-Black|Mobiles & Tablets|710         |710.0         |8451825.77   |
|MATSAM5A0BFFEF4DA20|Mobiles & Tablets|678         |678.0         |8031739.64   |
|MATINF5AE310D2D7A1A|Mobiles & Tablets|597         |597.0         |7371095.18   |
|MATINF59C9002EA6AF0|Mobiles & Tablets|634         |634.0         |7317707.35   |
|MATINF5A61FBB88

## Câu 3: Doanh thu theo tháng (Theo năm và tháng)
**Truy vấn:** Aggregation + GROUP BY nhiều cột (đa chiều: năm + tháng) + ORDER BY  
**Insight:** Tháng nào doanh thu cao nhất/thấp nhất trong từng năm - phục vụ lên kế hoạch
khuyến mãi theo mùa, dự trù nhân sự/kho bãi.


In [7]:
result3 = spark.sql("""
    SELECT
        order_year,
        order_month,
        COUNT(DISTINCT increment_id)            AS total_orders,
        SUM(qty_ordered)                        AS total_qty_sold,
        ROUND(SUM(grand_total), 2)              AS monthly_revenue,
        ROUND(AVG(grand_total), 2)              AS avg_order_value
    FROM sales
    WHERE order_year IS NOT NULL AND order_month IS NOT NULL
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""")
result3.show(36, truncate=False)

+----------+-----------+------------+--------------+---------------+---------------+
|order_year|order_month|total_orders|total_qty_sold|monthly_revenue|avg_order_value|
+----------+-----------+------------+--------------+---------------+---------------+
|2016      |7          |5727        |6942.0        |1.02666225E7   |1478.91        |
|2016      |8          |8226        |9524.0        |1.21126189E7   |1271.8         |
|2016      |9          |10153       |11980.0       |2.408061904E7  |2010.07        |
|2016      |10         |8217        |9829.0        |1.450246053E7  |1475.48        |
|2016      |11         |48023       |62340.0       |1.1202398165E8 |1796.98        |
|2016      |12         |8651        |10612.0       |2.355675713E7  |2219.82        |
|2017      |1          |7443        |9799.0        |2.721995791E7  |2777.83        |
|2017      |2          |6596        |9078.0        |2.414534325E7  |2659.76        |
|2017      |3          |10659       |15648.0       |4.649058661E7

## Câu 4: Tỷ lệ trạng thái đơn hàng (hoàn thành / hủy / hoàn trả)
**Truy vấn:** CASE WHEN + Aggregation + tính tỷ lệ %  
**Insight:** Tỷ lệ hủy đơn (`canceled`) và hoàn trả (`order_refunded`) chiếm bao nhiêu % -
nếu tỷ lệ cao, cần xem lại quy trình xác nhận đơn, vận chuyển, hoặc chất lượng sản phẩm.

In [8]:
result4 = spark.sql("""
    SELECT
        status,
        COUNT(DISTINCT increment_id)                                    AS total_orders,
        ROUND(SUM(grand_total), 2)                                      AS total_revenue,
        SUM(CASE WHEN status = 'complete'        THEN 1 ELSE 0 END)     AS completed,
        SUM(CASE WHEN status = 'canceled'        THEN 1 ELSE 0 END)     AS canceled,
        SUM(CASE WHEN status = 'order_refunded'  THEN 1 ELSE 0 END)     AS refunded,
        ROUND(
            COUNT(DISTINCT increment_id) * 100.0
            / SUM(COUNT(DISTINCT increment_id)) OVER ()
        , 2)                                                             AS pct_of_total_orders
    FROM sales
    GROUP BY status
    ORDER BY total_orders DESC
""")
result4.show(20, truncate=False)

+--------------+------------+--------------+---------+--------+--------+-------------------+
|status        |total_orders|total_revenue |completed|canceled|refunded|pct_of_total_orders|
+--------------+------------+--------------+---------+--------+--------+-------------------+
|complete      |140012      |4.9798012094E8|191172   |0       |0       |47.29              |
|canceled      |86673       |5.0667178082E8|0        |123348  |0       |29.27              |
|order_refunded|39135       |1.0340323977E8|0        |0       |46946   |13.22              |
|received      |25330       |1.6294919598E8|0        |0       |0       |8.55               |
|refund        |3577        |1.72524213E7  |0        |0       |0       |1.21               |
|cod           |800         |7039705.55    |0        |0       |0       |0.27               |
|closed        |274         |1467083.09    |0        |0       |0       |0.09               |
|paid          |238         |2421814.32    |0        |0       |0      

## Câu 5: Đơn hàng giá trị cao bị hủy hoặc hoàn trả
**Truy vấn:** Truy vấn điều kiện (WHERE) + Subquery (AVG làm ngưỡng) + CASE WHEN  
**Insight:** Các đơn hàng giá trị lớn (trên 2 lần giá trị trung bình) nhưng bị `canceled`/`order_refunded`
gây thất thoát doanh thu lớn - cần ưu tiên CSKH gọi xác nhận trước khi giao.

In [9]:
result5 = spark.sql("""
    SELECT
        increment_id,
        customer_id,
        category_name_1                         AS category,
        ROUND(grand_total, 2)                   AS grand_total,
        payment_method,
        status,
        CASE
            WHEN status = 'order_refunded' THEN 'Hoàn trả'
            WHEN status = 'canceled'       THEN 'Hủy đơn'
            ELSE 'Khác'
        END                                      AS status_group
    FROM sales
    WHERE status IN ('canceled', 'order_refunded')
      AND grand_total > (SELECT AVG(grand_total) * 2 FROM sales)
    ORDER BY grand_total DESC
    LIMIT 20
""")
result5.show(20, truncate=False)


+------------+-----------+-----------------+-----------+---------------+--------------+------------+
|increment_id|customer_id|category         |grand_total|payment_method |status        |status_group|
+------------+-----------+-----------------+-----------+---------------+--------------+------------+
|100317435   |7190       |Men's Fashion    |15928.05   |Payaxis        |canceled      |Hủy đơn     |
|100317409   |7190       |Men's Fashion    |15928.05   |Payaxis        |canceled      |Hủy đơn     |
|100317701   |7190       |Men's Fashion    |15928.05   |Payaxis        |canceled      |Hủy đơn     |
|100419888   |68583      |Mobiles & Tablets|15927.25   |Easypay        |order_refunded|Hoàn trả    |
|100418845   |76776      |Mobiles & Tablets|15927.25   |Easypay        |canceled      |Hủy đơn     |
|100421721   |23226      |Mobiles & Tablets|15927.25   |Easypay        |canceled      |Hủy đơn     |
|100416590   |70367      |Mobiles & Tablets|15927.25   |Easypay        |canceled      |Hủy 

## Câu 6: Top 10 khách hàng chi tiêu nhiều nhất (đơn đã hoàn thành)
**Truy vấn:** Aggregation + WHERE (truy vấn điều kiện) + GROUP BY + ORDER BY  
**Insight:** Nhóm khách hàng VIP - chi tiêu nhiều nhất với đơn hàng hoàn thành thành công,
là đối tượng ưu tiên cho chương trình khách hàng thân thiết.


In [10]:
result6 = spark.sql("""
    SELECT
        customer_id,
        COUNT(DISTINCT increment_id)            AS total_orders,
        SUM(qty_ordered)                        AS total_items_bought,
        ROUND(SUM(grand_total), 2)              AS total_spending,
        ROUND(AVG(grand_total), 2)              AS avg_order_value
    FROM sales
    WHERE status = 'complete'
    GROUP BY customer_id
    ORDER BY total_spending DESC
    LIMIT 10
""")
result6.show(10, truncate=False)

+-----------+------------+------------------+--------------+---------------+
|customer_id|total_orders|total_items_bought|total_spending|avg_order_value|
+-----------+------------+------------------+--------------+---------------+
|114        |522         |561.0             |1340330.95    |2389.18        |
|33         |651         |777.0             |1245289.35    |1602.69        |
|820        |732         |794.0             |1196240.6     |1506.6         |
|5769       |159         |174.0             |1190906.45    |6844.29        |
|1404       |576         |771.0             |1123010.97    |1456.56        |
|767        |638         |714.0             |1090278.82    |1527.0         |
|31025      |331         |540.0             |1087719.83    |2014.3         |
|36         |548         |591.0             |1068045.4     |1807.18        |
|64         |309         |321.0             |996878.53     |3105.54        |
|806        |595         |659.0             |968685.75     |1469.93        |

## Câu 7: Hiệu quả khuyến mãi theo danh mục — Tỷ lệ giảm giá vs doanh thu
**Truy vấn:** Aggregation + CASE WHEN phân nhóm mức giảm giá + GROUP BY đa chiều  
**Insight:** Danh mục nào đang giảm giá nhiều nhưng doanh thu không tương xứng -
xem xét lại chiến lược khuyến mãi để tránh giảm lợi nhuận không cần thiết.

In [11]:
result7 = spark.sql("""
    WITH base AS (
        SELECT
            category_name_1                                                     AS category,
            CASE
                WHEN discount_amount = 0                                                THEN 'Không giảm giá (0%)'
                WHEN discount_amount / (grand_total + discount_amount) <= 0.10          THEN 'Giảm nhẹ (≤10%)'
                WHEN discount_amount / (grand_total + discount_amount) <= 0.30          THEN 'Giảm vừa (10-30%)'
                ELSE                                                                         'Giảm sâu (>30%)'
            END                                                                     AS discount_tier,
            increment_id,
            grand_total,
            discount_amount
        FROM sales
        WHERE grand_total + discount_amount > 0
    )
    SELECT
        category,
        discount_tier,
        COUNT(DISTINCT increment_id)                                            AS total_orders,
        ROUND(SUM(grand_total), 2)                                              AS total_revenue,
        ROUND(SUM(discount_amount), 2)                                          AS total_discount,
        ROUND(
            SUM(discount_amount) * 100.0
            / NULLIF(SUM(grand_total + discount_amount), 0)
        , 2)                                                                     AS avg_discount_rate_pct,
        ROUND(AVG(grand_total), 2)                                              AS avg_order_value,
        -- CHỈ SỐ MỚI: mỗi đồng giảm giá thu về bao nhiêu doanh thu
        ROUND(
            SUM(grand_total)
            / NULLIF(SUM(discount_amount), 0)
        , 2)                                                                     AS revenue_per_discount
    FROM base
    GROUP BY category, discount_tier
    ORDER BY category, avg_discount_rate_pct
""")
result7.show(40, truncate=False)

+-----------------+-------------------+------------+--------------+--------------+---------------------+---------------+--------------------+
|category         |discount_tier      |total_orders|total_revenue |total_discount|avg_discount_rate_pct|avg_order_value|revenue_per_discount|
+-----------------+-------------------+------------+--------------+--------------+---------------------+---------------+--------------------+
|Appliances       |Không giảm giá (0%)|15470       |7.126635186E7 |0.0           |0.0                  |4277.44        |NULL                |
|Appliances       |Giảm nhẹ (≤10%)    |4749        |3.207152756E7 |2283712.16    |6.65                 |5855.67        |14.04               |
|Appliances       |Giảm vừa (10-30%)  |13223       |6.73937383E7  |1.451994232E7 |17.73                |4986.59        |4.64                |
|Appliances       |Giảm sâu (>30%)    |1953        |2963589.63    |5498218.83    |64.98                |1407.88        |0.54                |
|Beaut

## Câu 8: Tăng trưởng doanh thu theo tháng (so với tháng trước)
**Truy vấn:** Window Functions - LAG() + tính % tăng trưởng MoM  
**Insight:** Tháng nào tăng trưởng đột biến hoặc sụt giảm mạnh so với tháng trước -
giúp xác định nguyên nhân (mùa lễ, khuyến mãi, hoặc sự cố vận hành).


In [12]:
result8 = spark.sql("""
    WITH monthly AS (
        SELECT
            order_year,
            order_month,
            ROUND(SUM(grand_total), 2)          AS monthly_revenue
        FROM sales
        WHERE order_year IS NOT NULL AND order_month IS NOT NULL
        GROUP BY order_year, order_month
    )
    SELECT
        order_year,
        order_month,
        monthly_revenue,
        ROUND(LAG(monthly_revenue) OVER (
            ORDER BY order_year, order_month
        ), 2)                                    AS prev_month_revenue,
        ROUND(
            (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY order_year, order_month))
            * 100.0
            / NULLIF(LAG(monthly_revenue) OVER (ORDER BY order_year, order_month), 0)
        , 2)                                     AS mom_growth_pct
    FROM monthly
    ORDER BY order_year, order_month
""")
result8.show(36, truncate=False)

+----------+-----------+---------------+------------------+--------------+
|order_year|order_month|monthly_revenue|prev_month_revenue|mom_growth_pct|
+----------+-----------+---------------+------------------+--------------+
|2016      |7          |1.02666225E7   |NULL              |NULL          |
|2016      |8          |1.21126189E7   |1.02666225E7      |17.98         |
|2016      |9          |2.408061904E7  |1.21126189E7      |98.81         |
|2016      |10         |1.450246053E7  |2.408061904E7     |-39.78        |
|2016      |11         |1.1202398165E8 |1.450246053E7     |672.45        |
|2016      |12         |2.355675713E7  |1.1202398165E8    |-78.97        |
|2017      |1          |2.721995791E7  |2.355675713E7     |15.55         |
|2017      |2          |2.414534325E7  |2.721995791E7     |-11.3         |
|2017      |3          |4.649058661E7  |2.414534325E7     |92.54         |
|2017      |4          |4.01324039E7   |4.649058661E7     |-13.68        |
|2017      |5          |8

## Câu 9: Xếp hạng danh mục theo doanh thu mỗi năm (Top 3)
**Truy vấn:** Window Functions - RANK() OVER (PARTITION BY year)  
**Insight:** Danh mục nào luôn nằm top 3 mỗi năm (ổn định) và danh mục nào biến động -
hỗ trợ chiến lược đầu tư dài hạn theo danh mục.


In [13]:
result9 = spark.sql("""
    SELECT *
    FROM (
        SELECT
            order_year,
            category_name_1                                 AS category,
            ROUND(SUM(grand_total), 2)                      AS category_revenue,
            RANK() OVER (
                PARTITION BY order_year
                ORDER BY SUM(grand_total) DESC
            )                                                AS revenue_rank
        FROM sales
        WHERE order_year IS NOT NULL
        GROUP BY order_year, category_name_1
    )
    WHERE revenue_rank <= 3
    ORDER BY order_year, revenue_rank
""")
result9.show(30, truncate=False)

+----------+-----------------+----------------+------------+
|order_year|category         |category_revenue|revenue_rank|
+----------+-----------------+----------------+------------+
|2016      |Mobiles & Tablets|4.529222528E7   |1           |
|2016      |Appliances       |3.300043939E7   |2           |
|2016      |Men's Fashion    |3.130392187E7   |3           |
|2017      |Mobiles & Tablets|2.5016413564E8  |1           |
|2017      |Women's Fashion  |1.0995201186E8  |2           |
|2017      |Appliances       |9.568085192E7   |3           |
|2018      |Mobiles & Tablets|1.2449789293E8  |1           |
|2018      |Appliances       |4.501391605E7   |2           |
|2018      |Women's Fashion  |4.498525294E7   |3           |
+----------+-----------------+----------------+------------+



## Câu 10: Phương thức thanh toán nào hiệu quả nhất (doanh thu + tỷ lệ hủy)
**Truy vấn:** Aggregation + CASE WHEN (tính tỷ lệ hủy) + Phân tích đa chiều (đối chiếu nhiều chỉ số trên cùng GROUP BY)  
**Insight:** Phương thức thanh toán nào mang lại doanh thu cao nhất nhưng có tỷ lệ hủy thấp -
ưu tiên khuyến khích khách hàng sử dụng (ví dụ ưu đãi khi thanh toán online thay COD).


In [14]:
result10 = spark.sql("""
    SELECT
        payment_method,
        COUNT(DISTINCT increment_id)                                AS total_orders,
        ROUND(SUM(grand_total), 2)                                  AS total_revenue,
        ROUND(AVG(grand_total), 2)                                  AS avg_order_value,
        SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END)        AS canceled_orders,
        ROUND(
            SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) * 100.0
            / COUNT(DISTINCT increment_id)
        , 2)                                                        AS cancel_rate_pct
    FROM sales
    GROUP BY payment_method
    ORDER BY total_revenue DESC
""")
result10.show(20, truncate=False)

+-----------------+------------+--------------+---------------+---------------+---------------+
|payment_method   |total_orders|total_revenue |avg_order_value|canceled_orders|cancel_rate_pct|
+-----------------+------------+--------------+---------------+---------------+---------------+
|cod              |156379      |4.8863255017E8|2126.77        |14558          |9.31           |
|Payaxis          |43536       |2.5901537179E8|4092.19        |38774          |89.06          |
|Easypay          |35907       |2.3747975412E8|4567.45        |31209          |86.92          |
|easypay_voucher  |14520       |1.283854485E8 |7448.25        |6387           |43.99          |
|bankalfalah      |7579        |5.150686192E7 |4382.82        |7840           |103.44         |
|jazzwallet       |16731       |5.088329653E7 |1970.08        |11937          |71.35          |
|jazzvoucher      |8361        |3.758121676E7 |3593.54        |5416           |64.78          |
|Easypay_MA       |5880        |3.598929